# IPL Team Aliases — Detailed EDA, Cleaning & Export

## Objective

This notebook reviews the IPL team-alias lookup table before it is used to standardise team names in match or ball-by-ball analysis.

The workflow follows the same practical sequence used in a human-led data-cleaning review: inspect the source, ask focused quality questions, investigate unusual values, make only justified changes, validate the result, and export a dashboard-ready CSV.

**Scope:** structure, completeness, identifier quality, alias uniqueness, alias-to-team grouping, and conservative text cleaning.

**Important limitation:** this file alone cannot prove that every `team_id` matches the canonical team dimension. Cross-file key validation requires the relevant team master table.

## 1. Imports and Data Loading

In [25]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 80)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 100)

# Try the uploaded source filename first; keep the path easy to change if this
# notebook is moved into a project folder.
candidate_paths = [
    Path('team_aliases(1).csv'),
    Path('team_aliases.csv'),
    Path('../team_aliases(1).csv'),
    Path('../../team_aliases.csv'),
]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        'Could not find team_aliases CSV. Update DATA_PATH to the source file location.'
    )

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print(f'Source file: {DATA_PATH.resolve()}')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

Source file: C:\Users\Bhuvan\Desktop\IPL DATA\team_aliases.csv
Loaded: 46 rows × 3 columns


,alias_id,team_id,alias_name
0,1,1,RCB
1,2,1,Royal Challengers Bengaluru
2,3,252,Delhi Daredevils
3,4,494,Kings XI Punjab
4,5,1,Royal Challengers Bangalore


## 2. First Look — What Kind of Table Is This?

Before cleaning anything, establish the table's grain. Each row appears intended to represent one alias string connected to a team identifier. This is a lookup/bridge-style table, not a fact table of matches or deliveries.

In [26]:
display(df.head(10))
print('\nShape:', df.shape)
print('\nColumn names:', df.columns.tolist())
print('\nData types:')
display(df.dtypes.to_frame('dtype'))
print('\nDescriptive overview:')
display(df.describe(include='all').T)

,alias_id,team_id,alias_name
0,1,1,RCB
1,2,1,Royal Challengers Bengaluru
2,3,252,Delhi Daredevils
3,4,494,Kings XI Punjab
4,5,1,Royal Challengers Bangalore
5,6,1,Banglore
6,7,1,Bengaluru
7,8,252,Delhi Capitals
8,9,252,DC
9,10,252,DD



Shape: (46, 3)

Column names: ['alias_id', 'team_id', 'alias_name']

Data types:


,dtype
alias_id,int64
team_id,int64
alias_name,str



Descriptive overview:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
alias_id,46.0,NaN,NaN,NaN,23.5,13.422618,1.0,12.25,23.5,34.75,46.0
team_id,46.0,NaN,NaN,NaN,309.630435,444.732297,1.0,3.25,129.0,494.0,1419.0
alias_name,46,46,RCB,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Completeness and Exact Duplicate Check

The first practical question is whether any row is incomplete or repeated exactly. A lookup table is small, so even a single repeated or blank alias can affect name matching downstream.

In [27]:
null_summary = pd.DataFrame({
    'null_count': df.isna().sum(),
    'null_pct': (df.isna().mean() * 100).round(2),
})
display(null_summary)
print('Exact duplicate rows:', int(df.duplicated().sum()))

# Blank-string checks are separate from null checks: an empty string is not
# necessarily read by pandas as a null.
for col in df.select_dtypes(include='object').columns:
    blank_count = int(df[col].astype('string').str.strip().eq('').sum())
    print(f'{col}: whitespace-only / blank strings = {blank_count}')

,null_count,null_pct
alias_id,0,0.0
team_id,0,0.0
alias_name,0,0.0


Exact duplicate rows: 0
alias_name: whitespace-only / blank strings = 0


C:\Users\Bhuvan\AppData\Local\Temp\ipykernel_10944\363534816.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


## 4. Investigate the Identifier Columns

### 4.1 `alias_id` — Row Identifier

A surrogate key should normally be present and unique. Sequential numbering can be a useful descriptive observation, but downstream logic should rely on uniqueness rather than assuming that IDs must always be consecutive.

In [28]:
print('Missing alias_id:', int(df['alias_id'].isna().sum()))
print('Unique alias_id values:', int(df['alias_id'].nunique(dropna=True)))
print('Rows:', len(df))
print('alias_id unique:', bool(df['alias_id'].is_unique))
print('Minimum / maximum:', df['alias_id'].min(), '/', df['alias_id'].max())
expected_sequence = list(range(1, len(df) + 1))
print('Currently sequential from 1:', df['alias_id'].tolist() == expected_sequence)

duplicate_alias_ids = df[df['alias_id'].duplicated(keep=False)].sort_values('alias_id')
display(duplicate_alias_ids if not duplicate_alias_ids.empty else pd.DataFrame(columns=df.columns))

Missing alias_id: 0
Unique alias_id values: 46
Rows: 46
alias_id unique: True
Minimum / maximum: 1 / 46
Currently sequential from 1: True


,alias_id,team_id,alias_name


### 4.2 `team_id` — Grouping / Join Key

Multiple rows per `team_id` are expected: one franchise can have abbreviations, city names, and historical names. The useful checks are the number of represented team IDs and whether each group has been mapped consistently within this file.

In [29]:
team_counts = (
    df.groupby('team_id', dropna=False)
      .agg(alias_count=('alias_name', 'size'),
           distinct_aliases=('alias_name', 'nunique'))
      .sort_values(['alias_count', 'distinct_aliases'], ascending=False)
)
display(team_counts)
print('Distinct team_id values:', int(df['team_id'].nunique(dropna=True)))
print('Missing team_id values:', int(df['team_id'].isna().sum()))

,alias_count,distinct_aliases
team_id,,
1,5,5
252,5,5
2,4,4
4,4,4
494,4,4
3,3,3
6,3,3
129,3,3
134,3,3


Distinct team_id values: 14
Missing team_id values: 0


## 5. Investigate `alias_name`

For text used in joins, we need to distinguish harmless spelling variants from data-quality problems. Trimming surrounding whitespace is safe. Automatically correcting spellings or merging aliases is not safe without a trusted mapping decision.

In [30]:
alias_text = df['alias_name'].astype('string')
print('Missing alias_name:', int(df['alias_name'].isna().sum()))
print('Blank after trimming:', int(alias_text.str.strip().eq('').sum()))
print('Values with surrounding whitespace:',
      int((alias_text != alias_text.str.strip()).sum()))
print('Unique alias strings:', int(df['alias_name'].nunique(dropna=True)))

duplicate_alias_names = (
    df[df['alias_name'].duplicated(keep=False)]
      .sort_values('alias_name')
)
print('Rows involved in repeated alias_name values:', len(duplicate_alias_names))
display(duplicate_alias_names if not duplicate_alias_names.empty else
        pd.DataFrame(columns=df.columns))

Missing alias_name: 0
Blank after trimming: 0
Values with surrounding whitespace: 0
Unique alias strings: 46
Rows involved in repeated alias_name values: 0


,alias_id,team_id,alias_name


## 6. Review Aliases as Groups

The alias list is easier to reason about when viewed team by team. This is a review step, not an instruction to collapse the names into one canonical label: each variant may exist to recognise historical records in source data.

In [31]:
for team_id, group in df.sort_values(['team_id', 'alias_id']).groupby('team_id', dropna=False):
    print(f'team_id={team_id}: {group["alias_name"].tolist()}')

team_id=1: ['RCB', 'Royal Challengers Bengaluru', 'Royal Challengers Bangalore', 'Banglore', 'Bengaluru']
team_id=2: ['Sunrisers Hyderabad', 'SRH', 'Hyderabad', 'Deccan Chargers']
team_id=3: ['Mumbai Indians', 'MI', 'Mumbai']
team_id=4: ['Rising Pune Supergiant', 'Rising Pune Supergiants', 'RPS', 'Pune']
team_id=5: ['Gujarat Lions', 'GL']
team_id=6: ['Kolkata Knight Riders', 'KKR', 'Kolkatta']
team_id=129: ['Chennai Super Kings', 'CSK', 'Chennai']
team_id=134: ['Rajasthan Royals', 'Rajasthan', 'RR']
team_id=252: ['Delhi Daredevils', 'Delhi Capitals', 'DC', 'DD', 'Delhi']
team_id=494: ['Kings XI Punjab', 'Punjab Kings', 'KXIP', 'Punjab']
team_id=614: ['Lucknow Super Giants', 'LSG', 'Lucknow']
team_id=615: ['Gujarat Titans', 'GT']
team_id=1414: ['Kochi Tuskers Kerala', 'KTK', 'Kerala']
team_id=1419: ['Pune Warriors', 'PWI']


## 7. Specific Data-Quality Observations

A few entries deserve explicit documentation:

- `Banglore` and `Kolkatta` look like misspellings, but they may have been deliberately included to match messy historical/source text. Preserve them unless the project owner approves a canonicalisation rule.
- Singular/plural variants such as `Rising Pune Supergiant` and `Rising Pune Supergiants` may reflect naming variation.
- `Delhi Daredevils` / `Delhi Capitals` and `Kings XI Punjab` / `Punjab Kings` represent historical franchise naming changes in this lookup.
- `Deccan Chargers` is listed under `team_id = 2` in this file. The starter notebook notes that a separate team master may use a different ID for Deccan Chargers; that relationship cannot be verified from this CSV alone. Treat it as a cross-file modelling question, not something to silently rewrite here.

The key decision is to retain these values. The purpose of an alias table is often to preserve alternative strings, including imperfect spellings, so that they can be mapped consistently.

In [32]:
review_terms = ['Banglore', 'Kolkatta', 'Rising Pune Supergiant',
                'Rising Pune Supergiants', 'Delhi Daredevils',
                'Delhi Capitals', 'Kings XI Punjab', 'Punjab Kings',
                'Deccan Chargers']
review = df[df['alias_name'].isin(review_terms)].sort_values(['team_id', 'alias_id'])
display(review)

,alias_id,team_id,alias_name
5,6,1,Banglore
17,18,2,Deccan Chargers
21,22,4,Rising Pune Supergiant
22,23,4,Rising Pune Supergiants
29,30,6,Kolkatta
2,3,252,Delhi Daredevils
7,8,252,Delhi Capitals
3,4,494,Kings XI Punjab
11,12,494,Punjab Kings


## 8. Cleaning Decisions

Only deterministic, low-risk cleanup is applied:

1. Convert `alias_name` to pandas' nullable string dtype temporarily for text handling.
2. Strip leading/trailing whitespace if present.
3. Convert whitespace-only alias strings to missing values if any are present.
4. Do **not** drop rows, renumber identifiers, impute IDs, or merge alias spellings.
5. Preserve the source column order and row grain.

In the current source file, the inspection checks indicate no nulls, no blank strings, and no surrounding whitespace, so the cleaning step should leave the data values unchanged. The step remains in place for reproducibility if the source is refreshed.

In [33]:
rows_before = len(df)
columns_before = df.columns.tolist()

df['alias_name'] = df['alias_name'].astype('string').str.strip()
df['alias_name'] = df['alias_name'].mask(df['alias_name'].eq(''), pd.NA)

print('Rows before / after:', rows_before, '/', len(df))
print('Columns preserved:', df.columns.tolist() == columns_before)
print('Rows removed:', rows_before - len(df))
print('Whitespace-only aliases converted to missing:', int(df['alias_name'].isna().sum()))

Rows before / after: 46 / 46
Columns preserved: True
Rows removed: 0
Whitespace-only aliases converted to missing: 0


## 9. Final Validation

Re-check the conditions that matter for safe use in a dashboard or a name-normalisation step. These checks validate the file itself; they do not replace a join against the canonical team table.

In [34]:
validation = {
    'same row count as source': len(df) == len(df_raw),
    'same column order as source': df.columns.tolist() == df_raw.columns.tolist(),
    'alias_id has no missing values': df['alias_id'].notna().all(),
    'alias_id is unique': df['alias_id'].is_unique,
    'team_id has no missing values': df['team_id'].notna().all(),
    'alias_name has no missing values': df['alias_name'].notna().all(),
    'alias_name has no blank strings': not df['alias_name'].astype('string').str.strip().eq('').any(),
    'no exact duplicate rows': not df.duplicated().any(),
    'alias_name values are unique': df['alias_name'].is_unique,
}
validation_results = pd.DataFrame({
    'check': list(validation.keys()),
    'passed': list(validation.values()),
})
display(validation_results)
print('All validation checks passed:', bool(validation_results['passed'].all()))

,check,passed
0,same row count as source,True
1,same column order as source,True
2,alias_id has no missing values,True
3,alias_id is unique,True
4,team_id has no missing values,True
5,alias_name has no missing values,True
6,alias_name has no blank strings,True
7,no exact duplicate rows,True
8,alias_name values are unique,True


All validation checks passed: True


## 10. Export Cleaned CSV

Export without the pandas index so the CSV retains only the three intended fields. The output is written beside this notebook by default; update `OUTPUT_PATH` if your project has a dedicated processed-data folder.

In [35]:
from pathlib import Path

OUTPUT_DIR = Path("../../data/cleaned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "team_aliases_clean.csv"
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"Exported: {OUTPUT_PATH.resolve()}")
print(f"Export shape: {df.shape}")
print("Export columns:", df.columns.tolist())

df_check = pd.read_csv(OUTPUT_PATH)
print("Reloaded shape:", df_check.shape)
print("Reloaded columns match:", df_check.columns.tolist() == df.columns.tolist())
print("Reloaded exact duplicate rows:", int(df_check.duplicated().sum()))
display(df_check.head())

Exported: C:\Users\Bhuvan\Desktop\IPL DATA\data\cleaned\team_aliases_clean.csv
Export shape: (46, 3)
Export columns: ['alias_id', 'team_id', 'alias_name']
Reloaded shape: (46, 3)
Reloaded columns match: True
Reloaded exact duplicate rows: 0


,alias_id,team_id,alias_name
0,1,1,RCB
1,2,1,Royal Challengers Bengaluru
2,3,252,Delhi Daredevils
3,4,494,Kings XI Punjab
4,5,1,Royal Challengers Bangalore


## 11. EDA Findings Summary

In [36]:
summary = pd.DataFrame([
    ('Rows / columns', f'{df.shape[0]} rows × {df.shape[1]} columns'),
    ('Distinct team IDs', int(df['team_id'].nunique())),
    ('Missing values in source', int(df_raw.isna().sum().sum())),
    ('Exact duplicate rows in source', int(df_raw.duplicated().sum())),
    ('Unique alias IDs', bool(df['alias_id'].is_unique)),
    ('Unique alias names', bool(df['alias_name'].is_unique)),
    ('Rows dropped during cleaning', rows_before - len(df)),
    ('Cleaning applied', 'Trim surrounding whitespace; blank strings set to missing if present'),
    ('Canonicalisation', 'No alias spelling or franchise mapping changed'),
    ('Cross-file validation', 'Still requires the canonical team master table'),
], columns=['Item', 'Finding'])
display(summary)

,Item,Finding
0,Rows / columns,46 rows × 3 columns
1,Distinct team IDs,14
2,Missing values in source,0
3,Exact duplicate rows in source,0
4,Unique alias IDs,True
5,Unique alias names,True
6,Rows dropped during cleaning,0
7,Cleaning applied,Trim surrounding whitespace; blank strings set to missing if present
8,Canonicalisation,No alias spelling or franchise mapping changed
9,Cross-file validation,Still requires the canonical team master table


## Final recommendation

This source is suitable as a compact alias lookup after the notebook's validation checks pass. Keep all historical and alternate names in the alias table. If the analytics model needs one display name per franchise, maintain that as a separate canonical-team mapping rather than overwriting the aliases.

Before joining to a canonical team dimension, explicitly validate every `team_id` and resolve the noted `Deccan Chargers` mapping against the authoritative project data.